In [1]:
import os
import json
import pandas as pd
import numpy as np

In [2]:
def get_metadata_into_dataframe(parent_folder):
    
    all_metadata = []

    # Walk through the parent folder and its sub-directories
    for root, dirs, files in os.walk(parent_folder):
        for file in files:
            if file == "metadata.json":
                metadata_file_path = os.path.join(root, file)
                try:
                    with open(metadata_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        all_metadata.append(data)
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON from {metadata_file_path}: {e}")
                except Exception as e:
                    print(f"An error occurred while reading {metadata_file_path}: {e}")

    if all_metadata:
        # Create a DataFrame from the list of dictionaries
        df = pd.DataFrame(all_metadata)
        return df
    else:
        print(f"No 'metadata.json' files found in '{parent_folder}' or its subfolders.")
        return pd.DataFrame()

In [3]:
main_folder_path = 'data/OCR_Final' 

In [4]:
metadata_df = get_metadata_into_dataframe(main_folder_path)

In [5]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871


In [6]:
len(metadata_df)

46

In [7]:
split_genres = metadata_df['genre'].str.split(';', expand=True)

In [8]:
metadata_df['genre_main'] = split_genres[0].str.strip()

In [9]:
metadata_df['genre_sub'] = split_genres[1].str.strip().fillna('')

In [10]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682,Non-Fiction,Religious
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997,Non-Fiction,Poetry
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846,Non-Fiction,History
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987,Fiction,Religious
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871,Non-Fiction,


In [11]:
metadata_df['start_date'] = np.nan
metadata_df['end_date'] = np.nan

In [12]:
for index, row in metadata_df.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    metadata_df.at[index, 'start_date'] = start_year
    metadata_df.at[index, 'end_date'] = end_year


In [13]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub,start_date,end_date
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682,Non-Fiction,Religious,1187.0,1225.0
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997,Non-Fiction,Poetry,1901.0,1938.0
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846,Non-Fiction,History,1944.0,1944.0
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987,Fiction,Religious,1303.0,1333.0
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871,Non-Fiction,,1825.0,1905.0


In [14]:
len(metadata_df[metadata_df['end_date'] <= 1000])

1

In [15]:
len(metadata_df[(metadata_df['end_date'] > 1200) & (metadata_df['end_date'] < 1400)])

12

In [16]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [17]:
metadata_df['end_century'] = metadata_df['end_date'].apply(get_century)

In [18]:
metadata_df['end_century'].head()

0    13
1    20
2    20
3    14
4    20
Name: end_century, dtype: int64

In [19]:
metadata_df['end_century'].value_counts()

end_century
20    16
19     9
13     8
15     5
14     4
18     3
5      1
Name: count, dtype: int64

In [20]:
metadata_df['author'].value_counts()

author
Unknown                                                         14
මුනිදාස කුමාරතුංග                                                4
මාදම්පේ ධම්මතිලක හිමි                                            2
හික්කඩුවේ ශ්‍රි සුමංගල හිමි                                      2
රත්මලානේ ධර්මාලෝක හිමි                                           1
ශ්‍රීමද් බුද්ධදාස රජතුමා                                         1
දොන් අන්ද්‍රිස් සිල්වා                                           1
වැලිවිට සරණංකර සංඝරාජ හිමි                                       1
වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්මුල සංඝරාජ හිමි     1
මොග්ගල්ලාන හිමි                                                  1
ගුරුළු ගොමීන්                                                    1
වැලිකන්දේ ශ්‍රී සුමංගල හිමි                                      1
ඩි.එච්.එස් අභයරත්න                                               1
ඩී. ඊ. වික්‍රමසූරිය                                              1
තොටගමුවේ රාහුල හිමි                                    